<a href="https://colab.research.google.com/github/Rushikesh042/VLM-Dental-Classification/blob/main/Stage2_Biomarker_ProxyBranch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage II: Label-Blind Radiographic Bone-Support Biomarker (Proxy Branch)

Builds a formal, per-site bone-support biomarker from the frozen Stage-I teacher masks and
aggregates it to patient level. No clean reference and no CEJ landmarks exist, so this is
**Branch B**: a landmark-free proxy, explicitly not CEJ-to-crest millimetre bone loss. The
code keeps Branch A in place for when landmark annotations arrive.

**Firewall:** the biomarker consumes only image-derived masks, confidence, and geometry.
Weak labels, sources, age, sex, and record text are joined only afterwards, and only for a
post-hoc separation check against the reliable subsets, never inside the biomarker.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ---- stdlib ----
import json
import os
import pathlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*cuda.cudart.*")
warnings.filterwarnings("ignore", category=UserWarning, module="monai")

# ---- third-party ----
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.ndimage
import scipy.stats
import sklearn.metrics

PROJECT_ROOT = pathlib.Path(os.getenv("DENTAL_PROJECT_ROOT",
    "/content/drive/MyDrive/Teeth_Segmentation_Classification"))
PRE = PROJECT_ROOT / "Preprocessed_Dataset"
MMD_PRE = PRE / "MMDental"
MMD_TEACHER_DIR = MMD_PRE / "images_teacher_space"
MMD_MANIFEST = MMD_PRE / "manifests" / "manifest.csv"

BRIDGE = PROJECT_ROOT / "Stage1_Inference_MMDental"
TEACHER_SEG = BRIDGE / "teacher_seg"
GEOM_DIR = BRIDGE / "geometry"

STAGE_REST_ROOT = PROJECT_ROOT / "Stage2_to_Stage6_Biomarker_Arbiter_QA"
for d in ("biomarkers", "tables", "figures", "logs", "reports"):
    (STAGE_REST_ROOT / d).mkdir(parents=True, exist_ok=True)

def _optional_path(env_name):
    raw = os.getenv(env_name, "").strip()
    return pathlib.Path(raw) if raw else None

LANDMARK_REFERENCE_CSV = _optional_path("LANDMARK_REFERENCE_CSV")
USE_BRANCH_A = LANDMARK_REFERENCE_CSV is not None and LANDMARK_REFERENCE_CSV.exists()

TEACHER_SPACING = (0.4, 0.4, 0.4)
CONTACT_MM = float(os.getenv("STAGE2_CONTACT_MM", "2.0"))
SUPPORT_THR = float(os.getenv("STAGE2_SUPPORT_THR", "0.5"))
BONE_LEVEL_LOW = float(os.getenv("STAGE2_BONE_LEVEL_LOW", "0.6"))
N_BINS = int(os.getenv("STAGE2_N_BINS", "10"))
WORST_K = int(os.getenv("STAGE2_WORST_K", "3"))
SEED = int(os.getenv("STAGE2_SEED", "42"))
np.random.seed(SEED)

print("branch:", "A (CEJ-to-crest, landmarks present)" if USE_BRANCH_A else "B (landmark-free proxy)")
print("contact_mm:", CONTACT_MM, "| support_thr:", SUPPORT_THR, "| bone_level_low:", BONE_LEVEL_LOW)

branch: B (landmark-free proxy)
contact_mm: 2.0 | support_thr: 0.5 | bone_level_low: 0.6


In [3]:
# ---- fail loud on missing artifacts ----
REQUIRED = {"MMDental manifest": MMD_MANIFEST, "teacher_seg dir": TEACHER_SEG,
            "MMDental teacher images": MMD_TEACHER_DIR, "geometry dir": GEOM_DIR,}
missing = {k: str(v) for k, v in REQUIRED.items() if not v.exists()}
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" +
                            "\n".join(f"  {k}: {p}" for k, p in missing.items()))

available = sorted(p.name[:-len("_4class.npy")] for p in TEACHER_SEG.glob("*_4class.npy"))
assert available, "No teacher masks found; run the Part-3 bridge first."
print(f"{len(available)} cases with teacher masks")

403 cases with teacher masks


In [4]:
# ---- FIREWALL: biomarker inputs are masks/geometry only ----
FORBIDDEN_BIOMARKER_FIELDS = {
    "binary_label", "binary_label_clean", "periodontitis_label", "label",
    "label_source", "label_confidence", "record_label_quality", "evidence_source",
    "sample_weight", "age", "sex", "diagnosis_text", "treatment_text", "qa_answer"}

def assert_firewall(used_fields):
    leaked = set(used_fields) & FORBIDDEN_BIOMARKER_FIELDS
    if leaked:
        raise RuntimeError(f"FIREWALL VIOLATION: biomarker touched {sorted(leaked)}")

def load_spacing(pid):
    sidecar = GEOM_DIR / f"{pid}.json"

    if not sidecar.exists():
        raise FileNotFoundError(
            f"Missing geometry sidecar for patient {pid}: {sidecar}"
        )

    with open(sidecar, "r", encoding="utf-8") as f:
        geometry = json.load(f)

    if "spacing" not in geometry:
        raise KeyError(
            f"Missing spacing in geometry sidecar for patient {pid}"
        )

    spacing = tuple(
        float(x)
        for x in geometry["spacing"]
    )

    if len(spacing) != 3 or any(x <= 0 for x in spacing):
        raise ValueError(
            f"Invalid spacing for patient {pid}: {spacing}"
        )

    return spacing

print("firewall active;", len(FORBIDDEN_BIOMARKER_FIELDS), "forbidden fields")

firewall active; 14 forbidden fields


In [5]:
# ---- tested biomarker functions (label-blind), group-level aggregation ----
def surface_voxels(mask):
    mask = np.asarray(mask, dtype=bool)
    return mask & ~scipy.ndimage.binary_erosion(mask) if mask.any() else np.zeros_like(mask)

def _principal_axis(phys):
    centered = phys - phys.mean(0)
    _, vecs = np.linalg.eigh(np.cov(centered.T))
    return vecs[:, -1]

def tooth_site_biomarker(region, dist_to_jaw, spacing, contact_mm, n_bins, thr):
    """Per-site, height-resolved bone support. Used for the per-tooth table and heatmaps only."""
    coords = np.argwhere(region)
    if len(coords) < 30:
        return None
    phys = coords * np.array(spacing)
    axis = _principal_axis(phys)
    proj = phys @ axis
    proj = proj - proj.min()
    height = proj / max(proj.max(), 1e-6)
    adj = (dist_to_jaw[region] <= contact_mm).astype(float)
    lower = adj[height < 0.5].mean() if (height < 0.5).any() else 0.0
    upper = adj[height >= 0.5].mean() if (height >= 0.5).any() else 0.0
    if upper > lower:
        height = 1.0 - height               # apex (more bone) -> 0, crown -> 1
    bins = np.clip((height * n_bins).astype(int), 0, n_bins - 1)
    prof = np.array([adj[bins == b].mean() if (bins == b).any() else np.nan for b in range(n_bins)])
    supported = np.where(prof >= thr)[0]
    bone_level_frac = (supported.max() + 1) / n_bins if len(supported) else 0.0
    return {"support_iso": float(adj.mean()), "bone_level_frac": float(bone_level_frac),
            "crestal_support": float(np.nanmean(prof[int(n_bins * 0.66):])),
            "apical_support": float(np.nanmean(prof[:max(1, int(n_bins * 0.34))])),
            "n_vox": int(len(coords))}

GROUP_NAMES = {1: "incisor", 2: "canine", 3: "premolar", 4: "molar"}

def patient_biomarker(seg4, group_mask, conf4, spacing):
    """Patient biomarker. Per-tooth sites are kept for visualization, but the patient-level
    summaries are aggregated over the four TOOTH GROUPS, not over connected components.
    Group-level support is the feature that separated reliable cases in Step 0C (AUROC ~0.80);
    per-component minima saturate to zero on isolated mask fragments and destroy the signal."""
    teeth = seg4 == 1
    jaws = seg4 == 2
    if not teeth.any() or not jaws.any():
        return None, []
    dist_to_jaw = scipy.ndimage.distance_transform_edt(~jaws, sampling=spacing)
    adj_vol = dist_to_jaw <= CONTACT_MM

    # per-component sites: for the per-tooth table and heatmaps only (NOT patient aggregation)
    labelled, n = scipy.ndimage.label(teeth)
    sites = []
    for comp in range(1, n + 1):
        region = labelled == comp
        if region.sum() < 50:
            continue
        bm = tooth_site_biomarker(region, dist_to_jaw, spacing, CONTACT_MM, N_BINS, SUPPORT_THR)
        if bm is None:
            continue
        gv = group_mask[region]
        grp = int(np.bincount(gv[gv > 0]).argmax()) if (gv > 0).any() else 0
        bm["site_id"] = int(comp)
        bm["group"] = grp
        bm["group_name"] = GROUP_NAMES.get(grp, "unknown")
        sites.append(bm)
    sdf = pd.DataFrame(sites) if sites else pd.DataFrame()

    # group-level aggregation: robust support (0C-style) and a denoised per-group bone level
    group_support, group_bone = [], []
    for g in (1, 2, 3, 4):
        gm = (group_mask == g) & teeth
        if gm.sum() < 20:
            continue
        group_support.append(float((adj_vol & gm).sum()) / float(gm.sum()))
        if not sdf.empty and (sdf["group"] == g).any():
            group_bone.append(float(sdf.loc[sdf["group"] == g, "bone_level_frac"].median()))
    if not group_support:
        return None, sites
    gs = np.array(group_support)
    gb = np.array(group_bone) if group_bone else np.array([np.nan])
    k = min(WORST_K, len(gs))

    teeth_surf = surface_voxels(teeth)
    adj_surf = adj_vol & teeth_surf
    patient = {
        "n_groups": int(len(gs)),
        "n_sites": int(len(sites)),
        "min_support": float(gs.min()),
        "mean_support": float(gs.mean()),
        "lq_support": float(np.percentile(gs, 25)),
        "worstk_support": float(np.sort(gs)[:k].mean()),
        "min_bone_level": float(np.nanmin(gb)),
        "mean_bone_level": float(np.nanmean(gb)),
        "frac_groups_low_support": float((gs < SUPPORT_THR).mean()),
        "n_affected_groups": int(np.nansum(gb < BONE_LEVEL_LOW)),
        "boundary_contact_fraction": float(adj_surf.sum()) / max(int(teeth_surf.sum()), 1),
        "mean_jaw_conf_boundary": float(conf4[adj_surf].mean()) if (conf4 is not None and adj_surf.any()) else np.nan,
    }
    return patient, sites

print("biomarker functions ready (group-level aggregation)")

biomarker functions ready (group-level aggregation)


## Compute the biomarker over all MMDental cases

In [6]:
import shutil
import time

import tqdm.auto

LOCAL_SEG = pathlib.Path(os.getenv("STAGE2_LOCAL_SEG", "/content/stage2_teacher_seg"))
LOCAL_SEG.mkdir(parents=True, exist_ok=True)

def stage_local(pid):
    """Copy this case's masks/conf from Drive to local disk once; return local dir."""
    for suffix in ("_4class.npy", "_fdi.npy", "_4class_conf.npy", "_fdi_conf.npy"):
        src = TEACHER_SEG / f"{pid}{suffix}"
        dst = LOCAL_SEG / f"{pid}{suffix}"
        if src.exists() and (not dst.exists() or dst.stat().st_size != src.stat().st_size):
            shutil.copy2(src, dst)
    return LOCAL_SEG

def robust_load(path, retries=4, wait=2.0):
    last = None
    for attempt in range(retries):
        try:
            return np.load(path)
        except OSError as exc:           # transient Drive/FUSE read error
            last = exc
            time.sleep(wait * (attempt + 1))
    raise last

if USE_BRANCH_A:
    raise NotImplementedError("Branch A (CEJ-to-crest) requires the landmark loader; "
                              "landmarks were detected but the loader is not wired in this run.")

available = sorted(
    path.name.removesuffix("_4class.npy")
    for path in TEACHER_SEG.glob("*_4class.npy")
)

if not available:
    raise FileNotFoundError(
        f"No *_4class.npy files found in {TEACHER_SEG}"
    )

print(f"{len(available)} cases available for Stage II")
per_patient_rows, per_tooth_rows, skipped = [], [], []
progress = tqdm.auto.tqdm(available, desc="Stage II biomarker", unit="case")
for pid in progress:
    try:
        local = stage_local(pid)
        seg4 = robust_load(local / f"{pid}_4class.npy")
        grp = robust_load(local / f"{pid}_fdi.npy")          # 0..4 group codes (legacy filename)
        cpath = local / f"{pid}_4class_conf.npy"
        conf4 = robust_load(cpath).astype(np.float32) if cpath.exists() else None
    except OSError as exc:
        skipped.append(pid)
        progress.set_postfix(skipped=len(skipped))
        continue
    patient, sites = patient_biomarker(seg4, grp, conf4, load_spacing(pid))
    if patient is None:
        skipped.append(pid)
        progress.set_postfix(skipped=len(skipped))
        continue
    patient["patient_id"] = pid
    per_patient_rows.append(patient)
    for s in sites:
        s["patient_id"] = pid
        per_tooth_rows.append(s)

per_patient_df = pd.DataFrame(per_patient_rows)
per_tooth_df = pd.DataFrame(per_tooth_rows)
assert_firewall(per_patient_df.columns)
assert_firewall(per_tooth_df.columns)

per_patient_df.to_csv(STAGE_REST_ROOT / "biomarkers" / "per_patient_biomarkers.csv", index=False)
per_tooth_df.to_csv(STAGE_REST_ROOT / "biomarkers" / "per_tooth_biomarkers.csv", index=False)
pd.DataFrame({"patient_id": skipped}).to_csv(STAGE_REST_ROOT / "logs" / "stage2_skipped_cases.csv", index=False)

cfg = {"branch": "B_landmark_free_proxy", "contact_mm": CONTACT_MM, "support_thr": SUPPORT_THR,
       "bone_level_low": BONE_LEVEL_LOW, "n_bins": N_BINS, "worst_k": WORST_K,
       "n_patients": int(len(per_patient_df)), "n_sites": int(len(per_tooth_df)),
       "n_skipped": len(skipped), "teacher_spacing": list(TEACHER_SPACING)}
with open(STAGE_REST_ROOT / "biomarkers" / "stage2_config.json", "w") as f:
    json.dump(cfg, f, indent=2)
print(f"patients: {len(per_patient_df)} | sites: {len(per_tooth_df)} | skipped: {len(skipped)} {skipped}")

403 cases available for Stage II


Stage II biomarker:   0%|          | 0/403 [00:00<?, ?case/s]

patients: 402 | sites: 5583 | skipped: 1 ['585']


## Biomarker distributions and example per-site heatmaps

In [7]:
feat_cols = ["min_support", "mean_support", "worstk_support", "min_bone_level",
             "mean_bone_level", "frac_groups_low_support"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), feat_cols):
    ax.hist(per_patient_df[col].dropna(), bins=30)
    ax.set_title(col); ax.set_xlabel(col); ax.set_ylabel("patients")
plt.tight_layout()
plt.savefig(STAGE_REST_ROOT / "figures" / "biomarker_patient_distribution.png", dpi=120)
plt.close(fig)

# per-site bone-level heatmap for the 6 patients with the lowest min_bone_level
ex = per_patient_df.nsmallest(6, "min_bone_level")["patient_id"].tolist()
fig, axes = plt.subplots(len(ex), 1, figsize=(11, 1.5 * len(ex)))
if len(ex) == 1:
    axes = [axes]
for ax, pid in zip(axes, ex):
    sub = per_tooth_df[per_tooth_df["patient_id"] == pid].sort_values("group")
    vals = sub["bone_level_frac"].values[None, :]
    im = ax.imshow(vals, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_yticks([0]); ax.set_yticklabels([pid], fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["group_name"], rotation=90, fontsize=6)
fig.colorbar(im, ax=axes, fraction=0.02, label="bone level fraction")
plt.savefig(STAGE_REST_ROOT / "figures" / "biomarker_heatmap_examples.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("distribution and heatmap figures saved")

distribution and heatmap figures saved


## Post-hoc separation check against reliable subsets

This is a sanity check, not part of the biomarker. It joins the weak-label source only after
computation, to confirm the formal biomarker is at least as discriminative as the 0C proxy.

In [8]:
manifest = pd.read_csv(MMD_MANIFEST)
manifest["patient_id"] = manifest["patient_id"].astype(str)
source_col = next((c for c in ["label_source", "evidence_source"] if c in manifest.columns), None)
conf_col = next((c for c in ["record_label_quality", "label_confidence"] if c in manifest.columns), None)

def _to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )


def reliable_subsets(df):
    if {"reliable_pos", "reliable_neg"}.issubset(df.columns):
        return (
            _to_bool(df["reliable_pos"]),
            _to_bool(df["reliable_neg"]),
        )

    if source_col is None:
        return (
            pd.Series(False, index=df.index),
            pd.Series(False, index=df.index),
        )

    src = df[source_col].astype(str)
    pos = src.eq("icd_perio_positive")
    neg = src.isin(
        {
            "icd_gingivitis_or_gingival",
            "icd_other_only",
        }
    )

    if conf_col is not None:
        neg = neg & df[conf_col].astype(str).eq("high")

    return pos, neg

rel_pos, rel_neg = reliable_subsets(manifest)
lab = manifest[["patient_id"]].copy()
lab["reliable_pos"] = rel_pos.values
lab["reliable_neg"] = rel_neg.values
joined = per_patient_df.merge(lab, on="patient_id", how="inner")
pos_g = joined[joined["reliable_pos"]]
neg_g = joined[joined["reliable_neg"]]

# features where LOWER means more disease must be sign-flipped so higher score = positive
LOWER_IS_WORSE = {"min_support", "mean_support", "lq_support", "worstk_support",
                      "min_bone_level", "mean_bone_level"}
def cliffs_delta(a, b):
    a = np.asarray(a)[:, None]; b = np.asarray(b)[None, :]
    return float(((a > b).sum() - (a < b).sum()) / (a.shape[0] * b.shape[1]))

rows = []
if source_col is not None and len(pos_g) >= 5 and len(neg_g) >= 5:
    rng = np.random.default_rng(SEED)
    for feat in feat_cols + ["lq_support", "n_affected_groups"]:
        a = neg_g[feat].dropna().values
        b = pos_g[feat].dropna().values
        if len(a) < 3 or len(b) < 3:
            continue
        sign = -1.0 if feat in LOWER_IS_WORSE else 1.0
        y = np.r_[np.ones(len(b)), np.zeros(len(a))]
        s = np.r_[sign * b, sign * a]
        auroc = sklearn.metrics.roc_auc_score(y, s)
        boot = [sklearn.metrics.roc_auc_score(y[idx], s[idx])
                for idx in (rng.integers(0, len(y), len(y)) for _ in range(2000))
                if len(np.unique(y[idx])) == 2]
        lo, hi = np.percentile(boot, [2.5, 97.5]) if boot else (np.nan, np.nan)
        rows.append({"feature": feat, "n_pos": len(b), "n_neg": len(a),
                     "median_pos": float(np.median(b)), "median_neg": float(np.median(a)),
                     "auroc": float(auroc), "auroc_lo": float(lo), "auroc_hi": float(hi),
                     "mwu_p": float(scipy.stats.mannwhitneyu(b, a, alternative="two-sided").pvalue),
                     "cliffs_delta": cliffs_delta(b, a)})
    stats = pd.DataFrame(rows).drop_duplicates("feature").sort_values("auroc", ascending=False)
    stats.to_csv(STAGE_REST_ROOT / "tables" / "stage2_biomarker_reliable_subset_stats.csv", index=False)

    best = stats.iloc[0]
    a = neg_g[best["feature"]].dropna().values
    b = pos_g[best["feature"]].dropna().values
    sign = -1.0 if best["feature"] in LOWER_IS_WORSE else 1.0
    y = np.r_[np.ones(len(b)), np.zeros(len(a))]; s = np.r_[sign * b, sign * a]
    fpr, tpr, _ = sklearn.metrics.roc_curve(y, s)
    plt.figure(figsize=(5, 5))
    plt.plot(fpr, tpr, label=f"{best['feature']} AUROC {best['auroc']:.2f}")
    plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.legend(); plt.title("Stage II biomarker vs reliable subsets")
    plt.tight_layout(); plt.savefig(STAGE_REST_ROOT / "figures" / "stage2_biomarker_roc.png", dpi=120)
    plt.close()
    print(stats.to_string(index=False))
    print(f"\nbest: {best['feature']} AUROC {best['auroc']:.3f} [{best['auroc_lo']:.3f}, {best['auroc_hi']:.3f}]")
else:
    stats = pd.DataFrame()
    print("post-hoc check skipped: insufficient reliable cases or no source column")

                feature  n_pos  n_neg  median_pos  median_neg    auroc  auroc_lo  auroc_hi    mwu_p  cliffs_delta
           mean_support     23    290    0.460120    0.512786 0.769415  0.656468  0.868801 0.000017     -0.538831
         worstk_support     23    290    0.429038    0.485060 0.769415  0.648732  0.873459 0.000017     -0.538831
            min_support     23    290    0.385319    0.437141 0.761319  0.644047  0.860361 0.000030     -0.522639
             lq_support     23    290    0.413081    0.464794 0.749925  0.639290  0.850694 0.000066     -0.499850
frac_groups_low_support     23    290    0.750000    0.500000 0.717766  0.596555  0.825633 0.000185      0.435532
        mean_bone_level     23    290    0.666667    0.833333 0.645127  0.507787  0.772348 0.019617     -0.290255
      n_affected_groups     23    290    1.000000    0.000000 0.640855  0.513929  0.769270 0.010558      0.281709
         min_bone_level     23    290    0.500000    0.700000 0.579610  0.459978  0.6937

## Stage II report and status

In [9]:
best_line = ("not computed (no reliable subsets)" if stats.empty else
             f"{stats.iloc[0]['feature']} AUROC {stats.iloc[0]['auroc']:.3f} "
             f"[{stats.iloc[0]['auroc_lo']:.3f}, {stats.iloc[0]['auroc_hi']:.3f}]")
report = f"""# Stage II: label-blind bone-support biomarker (Branch B, proxy)

Patients: {len(per_patient_df)} | sites: {len(per_tooth_df)} | skipped: {len(skipped)}

This biomarker is a landmark-free proxy. It is NOT CEJ-to-crest millimetre bone loss, because
no CEJ landmarks are available. It measures, per tooth site, how far toward the crown the
alveolar bone still provides support, using only the frozen anatomy masks and geometry.

## Per-site features
support_iso, bone_level_frac (how far crownward bone reaches), crestal_support, apical_support.

## Patient aggregation
Lead feature is min_support, the worst-site support, following the Step 0C result. Also:
worst-k support and bone level, fraction of low-bone sites, affected-site count, and the
boundary jaw-confidence as a quality term.

## Post-hoc separation (sanity only, not part of the biomarker)
Best feature: {best_line}.
Step 0C crude proxy reference: AUROC 0.77. The Stage II biomarker should match or exceed it.

## Honesty and limits
Branch B is a proxy; impacted or fully embedded teeth read as healthy. Real bone-loss
calibration in millimetres needs CEJ landmarks (Branch A) and a clean reference for the
arbiter (Stage III), both currently absent.
"""
(STAGE_REST_ROOT / "reports" / "stage2_biomarker_report.md").write_text(report)

status = {"stage2_branch": "B_proxy", "n_patients": int(len(per_patient_df)),
          "n_sites": int(len(per_tooth_df)), "n_skipped": len(skipped),
          "best_posthoc_auroc": (None if stats.empty else float(stats.iloc[0]["auroc"])),
          "stage3_arbiter": "BLOCKED pending clean reference",
          "branch_a_cej": "BLOCKED pending landmarks"}
with open(STAGE_REST_ROOT / "reports" / "stage2_status.json", "w") as f:
    json.dump(status, f, indent=2)
print(json.dumps(status, indent=2))

{
  "stage2_branch": "B_proxy",
  "n_patients": 402,
  "n_sites": 5583,
  "n_skipped": 1,
  "best_posthoc_auroc": 0.7694152923538231,
  "stage3_arbiter": "BLOCKED pending clean reference",
  "branch_a_cej": "BLOCKED pending landmarks"
}
